# Canadian Bank Credit Stress Test - walkthrough

Big 6 Canadian lending, three portfolios, three 9-quarter scenarios, two loss methods (macro regression + Vasicek), CET1 impact and ranking.

Run `make all` first, or execute this notebook top to bottom.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from src import config, capital, charts, vasicek
from src.load_macro import load_macro, unemp_sigma
from src.load_banks import load_and_build
from src.regression import fit_all, stability_table
from src.scenarios import build_scenarios, jump_off_levels, scenario_summary
pd.set_option('display.width', 160)

## 1. Data
Macro: BoC Valet, StatCan, BIS house prices. Banks: hand-collected SFI figures mapped to three portfolios (`data/portfolio_map.csv`).

In [ ]:
macro = load_macro()
panel, cap = load_and_build()
panel.groupby(['bank', 'portfolio'])['pcl_bps'].describe().round(1)

In [ ]:
charts.pcl_history(panel);
charts.pcl_vs_macro(panel, macro);

## 2. Satellite regression
Bank fixed effects, lagged PCL, macro drivers kept only when correctly signed. Check: signs, size of coefficients, stability across sub-periods.

In [ ]:
models = fit_all(panel, macro)
for p, m in models.items():
    print(f'\n{p}: n={m.nobs}, R2={m.rsquared:.2f}, dropped={m.dropped}')
    display(m.summary_frame().round(3))
    display(stability_table(panel, macro, m).round(3))

## 3. Scenarios

In [ ]:
jo = capital.jump_off_quarter(panel)
scen = build_scenarios(macro, jo)
start = jump_off_levels(macro, jo)
scenario_summary(scen, start).round(2)

## 4. Projection, Vasicek cross-check, CET1 impact

In [ ]:
out = capital.run(models, scen, panel, cap, macro, unemp_sigma(macro), method='average')
capital.method_agreement(out).round(0)

In [ ]:
for p in config.PORTFOLIOS:
    print(p); display(vasicek.sensitivity_table(p, scen['stagflation'], start.unemp, start.hpi, unemp_sigma(macro)).round(0))

In [ ]:
charts.loss_paths(out['losses'], panel);
charts.cet1_paths(out['path'], 'stagflation');

## 5. Ranking

In [ ]:
capital.rank(out['summary'], 'stagflation')[['name', 'cet1_start', 'cet1_low', 'drawdown_bp', 'buffer_vs_osfi_bp', 'main_driver']]

In [ ]:
capital.ranking_by_method(out)

In [ ]:
bridges = {b: capital.waterfall(out['path'], out['capital_inputs'], out['losses'], b, 'stagflation') for b in out['capital_inputs'].index}
charts.cet1_waterfall(bridges, 'stagflation');
charts.loss_decomposition(out['summary'], 'stagflation');

## 6. Benchmark against banks' IFRS 9 disclosures
Requires `data/raw/banks/ifrs9_scenarios.csv` and `ifrs9_sensitivity.csv`.

In [ ]:
from src import benchmark
benchmark.scenario_severity(scenario_summary(scen, start))

In [ ]:
benchmark.rank_agreement(out['summary'], 'adverse')